In [4]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# ---- NSL-KDD — Random Forest ----
train_df_nsl = pd.read_csv(r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\NSL-KDD\processed\NSLKDD_train_cleaned.csv")
test_df_nsl = pd.read_csv(r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\NSL-KDD\processed\NSLKDD_test_cleaned.csv")

X_train_nsl = train_df_nsl.drop(columns=['label', 'binary_label', 'difficulty'])
y_train_nsl = train_df_nsl['binary_label']
X_test_nsl = test_df_nsl.drop(columns=['label', 'binary_label', 'difficulty'])
y_test_nsl = test_df_nsl['binary_label']
attack_labels_nsl = test_df_nsl['label']  # specific attack type, e.g. 'neptune', 'satan'

model_nsl = RandomForestClassifier(random_state=42)
model_nsl.fit(X_train_nsl, y_train_nsl)
y_pred_nsl = model_nsl.predict(X_test_nsl)

# Recall per specific attack category (only rows that are actually attacks)
results_nsl = []
attack_mask = y_test_nsl == 'attack'
for category in sorted(attack_labels_nsl[attack_mask].unique()):
    cat_mask = attack_mask & (attack_labels_nsl == category)
    total = cat_mask.sum()
    if total == 0:
        continue
    caught = (y_pred_nsl[cat_mask.values] == 'attack').sum()
    recall = caught / total
    results_nsl.append({'attack_category': category, 'count': total, 'recall': round(recall, 4)})

nsl_category_df = pd.DataFrame(results_nsl).sort_values('count', ascending=False)
print("NSL-KDD — Random Forest recall by attack category (unseen test set):")
print(nsl_category_df.to_string(index=False))

# ---- UNSW-NB15 — Random Forest ----
train_df_unsw = pd.read_csv(r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\UNSW-NB15\processed\UNSWNB15_train_cleaned.csv")
test_df_unsw = pd.read_csv(r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\UNSW-NB15\processed\UNSWNB15_test_cleaned.csv")

X_train_unsw = train_df_unsw.drop(columns=['label', 'attack_cat'])
y_train_unsw = train_df_unsw['label']
X_test_unsw = test_df_unsw.drop(columns=['label', 'attack_cat'])
y_test_unsw = test_df_unsw['label']
attack_cat_unsw = test_df_unsw['attack_cat']

model_unsw = RandomForestClassifier(random_state=42)
model_unsw.fit(X_train_unsw, y_train_unsw)
y_pred_unsw = model_unsw.predict(X_test_unsw)

results_unsw = []
attack_mask = y_test_unsw == 1
for category in sorted(attack_cat_unsw[attack_mask].dropna().unique()):
    cat_mask = attack_mask & (attack_cat_unsw == category)
    total = cat_mask.sum()
    if total == 0:
        continue
    caught = (y_pred_unsw[cat_mask.values] == 1).sum()
    recall = caught / total
    results_unsw.append({'attack_category': category, 'count': total, 'recall': round(recall, 4)})

unsw_category_df = pd.DataFrame(results_unsw).sort_values('count', ascending=False)
print("\nUNSW-NB15 — Random Forest recall by attack category (unseen test set):")
print(unsw_category_df.to_string(index=False))

# ---- CICIDS2017 — Decision Tree ----
train_df_cic = pd.read_csv(r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\CIC-IDS2017\processed\CICIDS2017_train_cleaned.csv")
test_df_cic = pd.read_csv(r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\CIC-IDS2017\processed\CICIDS2017_test_cleaned.csv")

X_train_cic = train_df_cic.drop(columns=['Label', 'binary_label'])
y_train_cic = train_df_cic['binary_label']
X_test_cic = test_df_cic.drop(columns=['Label', 'binary_label'])
y_test_cic = test_df_cic['binary_label']
attack_label_cic = test_df_cic['Label']  # specific attack type

model_cic = DecisionTreeClassifier(random_state=42)
model_cic.fit(X_train_cic, y_train_cic)
y_pred_cic = model_cic.predict(X_test_cic)

results_cic = []
attack_mask = y_test_cic == 'ATTACK'
for category in sorted(attack_label_cic[attack_mask].unique()):
    cat_mask = attack_mask & (attack_label_cic == category)
    total = cat_mask.sum()
    if total == 0:
        continue
    caught = (y_pred_cic[cat_mask.values] == 'ATTACK').sum()
    recall = caught / total
    results_cic.append({'attack_category': category, 'count': total, 'recall': round(recall, 4)})

cic_category_df = pd.DataFrame(results_cic).sort_values('count', ascending=False)
print("\nCICIDS2017 — Decision Tree recall by attack category (unseen test set):")
print(cic_category_df.to_string(index=False))

# ---- SAVE ----
import os
results_folder = r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\Phase3b_results"
os.makedirs(results_folder, exist_ok=True)
nsl_category_df.to_csv(os.path.join(results_folder, "AttackCategory_NSLKDD.csv"), index=False)
unsw_category_df.to_csv(os.path.join(results_folder, "AttackCategory_UNSWNB15.csv"), index=False)
cic_category_df.to_csv(os.path.join(results_folder, "AttackCategory_CICIDS2017.csv"), index=False)
print("\nSaved successfully!")

NSL-KDD — Random Forest recall by attack category (unseen test set):
attack_category  count  recall
        neptune   4657  0.9989
   guess_passwd   1231  0.0000
          mscan    996  0.6345
    warezmaster    944  0.1684
        apache2    737  0.1669
          satan    735  0.9986
   processtable    685  0.0730
          smurf    665  0.1940
           back    359  0.9554
      snmpguess    331  0.0000
          saint    319  0.7085
       mailbomb    293  0.0000
  snmpgetattack    178  0.0000
      portsweep    157  1.0000
        ipsweep    141  0.1560
     httptunnel    133  0.7820
           nmap     73  1.0000
            pod     41  0.8293
buffer_overflow     20  0.0000
       multihop     18  0.1111
          named     17  0.1765
             ps     15  0.0000
       sendmail     14  0.0000
          xterm     13  0.2308
        rootkit     13  0.0000
       teardrop     12  1.0000
          xlock      9  0.0000
           land      7  0.7143
         xsnoop      4  0.2500
 

In [5]:
# ---- STEP 2 (FAST VERSION): Per-attack-category recall under each imbalance-handling condition ----

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.linear_model import LogisticRegression
import time

attack_mask_cic = y_test_cic == 'ATTACK'
categories_cic = sorted(attack_labels_cic[attack_mask_cic].unique())

print("Applying SMOTE (parallelised)...")
t0 = time.time()
smote = SMOTE(random_state=42, n_jobs=-1)
X_train_cic_smote, y_train_cic_smote = smote.fit_resample(X_train_cic, y_train_cic)
print(f"SMOTE done in {time.time()-t0:.1f}s.")

print("Applying random undersampling...")
t0 = time.time()
undersampler = RandomUnderSampler(random_state=42)
X_train_cic_under, y_train_cic_under = undersampler.fit_resample(X_train_cic, y_train_cic)
print(f"Undersampling done in {time.time()-t0:.1f}s.")

conditions = {
    'Baseline (No Handling)': (X_train_cic, y_train_cic, None),
    'SMOTE': (X_train_cic_smote, y_train_cic_smote, None),
    'Random Undersampling': (X_train_cic_under, y_train_cic_under, None),
    'Class Weighting': (X_train_cic, y_train_cic, 'balanced'),
}

step2_results = []

for condition_name, (X_tr, y_tr, class_weight) in conditions.items():
    print(f"\n{'='*50}\nCONDITION: {condition_name}\n{'='*50}")

    dt_kwargs = {'random_state': 42}
    if class_weight:
        dt_kwargs['class_weight'] = class_weight
    dt_model = DecisionTreeClassifier(**dt_kwargs)

    lr_model = LogisticRegression(max_iter=1000, random_state=42, class_weight=class_weight, n_jobs=-1)

    for model_name, model in [('Decision Tree', dt_model), ('Logistic Regression', lr_model)]:
        t0 = time.time()
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_test_cic)
        print(f"  {model_name} fitted in {time.time()-t0:.1f}s.")

        for category in categories_cic:
            cat_mask = attack_mask_cic & (attack_labels_cic == category)
            total = cat_mask.sum()
            if total == 0:
                continue
            caught = (y_pred[cat_mask.values] == 'ATTACK').sum()
            recall = caught / total
            step2_results.append({
                'condition': condition_name,
                'model': model_name,
                'attack_category': category,
                'count': total,
                'recall': round(recall, 4)
            })

step2_df = pd.DataFrame(step2_results)
step2_df.to_csv(os.path.join(save_folder, "Phase3b_CICIDS2017_condition_x_category_recall.csv"), index=False)
print("\nStep 2 saved.")

pd.set_option('display.max_rows', None)
print(step2_df.pivot_table(index='attack_category', columns=['model', 'condition'], values='recall').round(3))

NameError: name 'attack_labels_cic' is not defined

In [6]:
train_df_cic = pd.read_csv(r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\CIC-IDS2017\processed\CICIDS2017_train_cleaned.csv")
test_df_cic = pd.read_csv(r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\CIC-IDS2017\processed\CICIDS2017_test_cleaned.csv")

X_train_cic = train_df_cic.drop(columns=['Label', 'binary_label'])
y_train_cic = train_df_cic['binary_label']
X_test_cic = test_df_cic.drop(columns=['Label', 'binary_label'])
y_test_cic = test_df_cic['binary_label']
attack_labels_cic = test_df_cic['Label']

import os
save_folder = r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\Phase3b_results"
os.makedirs(save_folder, exist_ok=True)

print("Reloaded. Shapes:", X_train_cic.shape, X_test_cic.shape)

Reloaded. Shapes: (2016638, 78) (504160, 78)


In [7]:
# ---- STEP 2 (FAST VERSION): Per-attack-category recall under each imbalance-handling condition ----

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
import time

attack_mask_cic = y_test_cic == 'ATTACK'
categories_cic = sorted(attack_labels_cic[attack_mask_cic].unique())

print("Applying SMOTE (parallelised)...")
t0 = time.time()
smote = SMOTE(random_state=42, n_jobs=-1)
X_train_cic_smote, y_train_cic_smote = smote.fit_resample(X_train_cic, y_train_cic)
print(f"SMOTE done in {time.time()-t0:.1f}s.")

print("Applying random undersampling...")
t0 = time.time()
undersampler = RandomUnderSampler(random_state=42)
X_train_cic_under, y_train_cic_under = undersampler.fit_resample(X_train_cic, y_train_cic)
print(f"Undersampling done in {time.time()-t0:.1f}s.")

conditions = {
    'Baseline (No Handling)': (X_train_cic, y_train_cic, None),
    'SMOTE': (X_train_cic_smote, y_train_cic_smote, None),
    'Random Undersampling': (X_train_cic_under, y_train_cic_under, None),
    'Class Weighting': (X_train_cic, y_train_cic, 'balanced'),
}

step2_results = []

for condition_name, (X_tr, y_tr, class_weight) in conditions.items():
    print(f"\n{'='*50}\nCONDITION: {condition_name}\n{'='*50}")

    dt_kwargs = {'random_state': 42}
    if class_weight:
        dt_kwargs['class_weight'] = class_weight
    dt_model = DecisionTreeClassifier(**dt_kwargs)

    lr_model = LogisticRegression(max_iter=1000, random_state=42, class_weight=class_weight, n_jobs=-1)

    for model_name, model in [('Decision Tree', dt_model), ('Logistic Regression', lr_model)]:
        t0 = time.time()
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_test_cic)
        print(f"  {model_name} fitted in {time.time()-t0:.1f}s.")

        for category in categories_cic:
            cat_mask = attack_mask_cic & (attack_labels_cic == category)
            total = cat_mask.sum()
            if total == 0:
                continue
            caught = (y_pred[cat_mask.values] == 'ATTACK').sum()
            recall = caught / total
            step2_results.append({
                'condition': condition_name,
                'model': model_name,
                'attack_category': category,
                'count': total,
                'recall': round(recall, 4)
            })

step2_df = pd.DataFrame(step2_results)
step2_df.to_csv(os.path.join(save_folder, "Phase3b_CICIDS2017_condition_x_category_recall.csv"), index=False)
print("\nStep 2 saved.")

pd.set_option('display.max_rows', None)
print(step2_df.pivot_table(index='attack_category', columns=['model', 'condition'], values='recall').round(3))

Applying SMOTE (parallelised)...


TypeError: SMOTE.__init__() got an unexpected keyword argument 'n_jobs'

In [8]:
# ---- STEP 2 (FIXED): Per-attack-category recall under each imbalance-handling condition ----

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
import time

attack_mask_cic = y_test_cic == 'ATTACK'
categories_cic = sorted(attack_labels_cic[attack_mask_cic].unique())

print("Applying SMOTE...")
t0 = time.time()
smote = SMOTE(random_state=42)
X_train_cic_smote, y_train_cic_smote = smote.fit_resample(X_train_cic, y_train_cic)
print(f"SMOTE done in {time.time()-t0:.1f}s.")

print("Applying random undersampling...")
t0 = time.time()
undersampler = RandomUnderSampler(random_state=42)
X_train_cic_under, y_train_cic_under = undersampler.fit_resample(X_train_cic, y_train_cic)
print(f"Undersampling done in {time.time()-t0:.1f}s.")

conditions = {
    'Baseline (No Handling)': (X_train_cic, y_train_cic, None),
    'SMOTE': (X_train_cic_smote, y_train_cic_smote, None),
    'Random Undersampling': (X_train_cic_under, y_train_cic_under, None),
    'Class Weighting': (X_train_cic, y_train_cic, 'balanced'),
}

step2_results = []

for condition_name, (X_tr, y_tr, class_weight) in conditions.items():
    print(f"\n{'='*50}\nCONDITION: {condition_name}\n{'='*50}")

    dt_kwargs = {'random_state': 42}
    if class_weight:
        dt_kwargs['class_weight'] = class_weight
    dt_model = DecisionTreeClassifier(**dt_kwargs)

    lr_model = LogisticRegression(max_iter=1000, random_state=42, class_weight=class_weight, n_jobs=-1)

    for model_name, model in [('Decision Tree', dt_model), ('Logistic Regression', lr_model)]:
        t0 = time.time()
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_test_cic)
        print(f"  {model_name} fitted in {time.time()-t0:.1f}s.")

        for category in categories_cic:
            cat_mask = attack_mask_cic & (attack_labels_cic == category)
            total = cat_mask.sum()
            if total == 0:
                continue
            caught = (y_pred[cat_mask.values] == 'ATTACK').sum()
            recall = caught / total
            step2_results.append({
                'condition': condition_name,
                'model': model_name,
                'attack_category': category,
                'count': total,
                'recall': round(recall, 4)
            })

step2_df = pd.DataFrame(step2_results)
step2_df.to_csv(os.path.join(save_folder, "Phase3b_CICIDS2017_condition_x_category_recall.csv"), index=False)
print("\nStep 2 saved.")

pd.set_option('display.max_rows', None)
print(step2_df.pivot_table(index='attack_category', columns=['model', 'condition'], values='recall').round(3))

Applying SMOTE...
SMOTE done in 299.1s.
Applying random undersampling...
Undersampling done in 11.5s.

CONDITION: Baseline (No Handling)
  Decision Tree fitted in 236.5s.
  Logistic Regression fitted in 125.2s.

CONDITION: SMOTE
  Decision Tree fitted in 379.0s.
  Logistic Regression fitted in 274.5s.

CONDITION: Random Undersampling
  Decision Tree fitted in 54.0s.
  Logistic Regression fitted in 45.5s.

CONDITION: Class Weighting
  Decision Tree fitted in 225.5s.
  Logistic Regression fitted in 123.8s.

Step 2 saved.
model                               Decision Tree                  \
condition                  Baseline (No Handling) Class Weighting   
attack_category                                                     
Bot                                         0.830           0.809   
DDoS                                        1.000           1.000   
DoS GoldenEye                               1.000           0.998   
DoS Hulk                                    0.999           0

In [9]:
step2_df = pd.read_csv(r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\Phase3b_results\Phase3b_CICIDS2017_condition_x_category_recall.csv")
pd.set_option('display.width', 200)
for model in step2_df['model'].unique():
    print(f"\n=== {model} ===")
    print(step2_df[step2_df['model']==model].pivot(index='attack_category', columns='condition', values='recall').round(3))


=== Decision Tree ===
condition                   Baseline (No Handling)  Class Weighting  Random Undersampling  SMOTE
attack_category                                                                                 
Bot                                          0.830            0.809                 0.929  0.880
DDoS                                         1.000            1.000                 1.000  1.000
DoS GoldenEye                                1.000            0.998                 0.999  0.998
DoS Hulk                                     0.999            0.999                 0.999  0.999
DoS Slowhttptest                             0.992            0.997                 0.999  0.999
DoS slowloris                                0.993            0.995                 0.995  0.995
FTP-Patator                                  1.000            0.999                 0.999  1.000
Heartbleed                                   0.000            0.000                 1.000  0.000
Infiltr

In [10]:
# Pull attack category counts from Step 1's result and merge into Step 2's table
cic_counts = cic_category_df[['attack_category', 'count']]

step2_with_counts = step2_df.merge(cic_counts, on='attack_category', how='left')
step2_with_counts.to_csv(os.path.join(save_folder, "Phase3b_CICIDS2017_condition_x_category_recall_with_counts.csv"), index=False)

for model in step2_with_counts['model'].unique():
    print(f"\n=== {model} ===")
    sub = step2_with_counts[step2_with_counts['model']==model]
    pivot = sub.pivot(index='attack_category', columns='condition', values='recall').round(3)
    pivot.insert(0, 'count', sub.groupby('attack_category')['count'].first())
    pivot = pivot.sort_values('count', ascending=False)
    print(pivot)


=== Decision Tree ===


KeyError: 'Column not found: count'

In [11]:
for model in step2_df['model'].unique():
    print(f"\n=== {model} ===")
    sub = step2_df[step2_df['model']==model]
    pivot = sub.pivot(index='attack_category', columns='condition', values='recall').round(3)
    pivot.insert(0, 'count', sub.groupby('attack_category')['count'].first())
    pivot = pivot.sort_values('count', ascending=False)
    print(pivot)


=== Decision Tree ===
condition                   count  Baseline (No Handling)  Class Weighting  Random Undersampling  SMOTE
attack_category                                                                                        
DoS Hulk                    34511                   0.999            0.999                 0.999  0.999
DDoS                        25770                   1.000            1.000                 1.000  1.000
PortScan                    18163                   0.989            0.989                 0.998  0.991
DoS GoldenEye                2046                   1.000            0.998                 0.999  0.998
FTP-Patator                  1146                   1.000            0.999                 0.999  1.000
DoS slowloris                1048                   0.993            0.995                 0.995  0.995
DoS Slowhttptest              984                   0.992            0.997                 0.999  0.999
SSH-Patator                   654        